# Benchmark Suite -- Cloud Runner

## Paper Title: *"A Zero-Allocation Dzul's GET-NFI Constructive Heuristic with Candidate-Set 2-Opt for the Travelling Salesperson Problem"*

> **Reproducibility Note**
>
> This notebook is designed for **Kaggle, Google Colab, or local execution**.
> It clones the repository, installs all dependencies (Python + Rust), runs the
> complete benchmark pipeline, and displays the results inline.
>
> Expected total runtime: **30-60 minutes** (dominated by Rust compilation and
> Divan microbenchmarks). Results are cached in scripts/materials/ so
> re-running the viewer cells skips the pipeline.

### Run Order
1. **Cell 1** -- Detect environment (Kaggle / Colab / local) and set up paths.
2. **Cell 2** -- Clone the repository (skipped if already present).
3. **Cell 3** -- Install Python dependencies and the Rust toolchain.
4. **Cell 4** -- Execute the full benchmark pipeline.
5. **Cells 5-8** -- View results and generate publication figures.
6. **Cell 9** -- Package results for download.


In [ ]:
import os
import sys
from pathlib import Path


def _detect_environment():
    """Detect the execution environment and resolve the repository root.

    Returns:
        A dictionary with keys ``env`` (``"kaggle"`` | ``"colab"`` |
        ``"local"``), ``repo_root`` (pathlib.Path), and
        ``is_cloud`` (bool).
    """
    # --- Kaggle detection ---
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        base = Path("/kaggle/working")
        repo = base / "dzul-get-nfi"
        return {"env": "kaggle", "repo_root": repo, "is_cloud": True}

    # --- Colab detection ---
    try:
        import google.colab  # noqa: F401
        base = Path("/content")
        repo = base / "dzul-get-nfi"
        return {"env": "colab", "repo_root": repo, "is_cloud": True}
    except ImportError:
        pass

    # --- Local ---
    cwd = Path.cwd()
    if (cwd / "Cargo.toml").exists():
        return {"env": "local", "repo_root": cwd, "is_cloud": False}
    if (cwd.parent / "Cargo.toml").exists():
        return {"env": "local", "repo_root": cwd.parent, "is_cloud": False}
    return {"env": "local", "repo_root": cwd, "is_cloud": False}


ENV = _detect_environment()
REPO_ROOT = ENV["repo_root"]
MATERIALS_DIR = REPO_ROOT / "scripts" / "materials"
PLOTS_DIR = REPO_ROOT / "scripts" / "plots"
IS_CLOUD = ENV["is_cloud"]

print(f"  Environment : {ENV['env']}")
print(f"  Repo root   : {REPO_ROOT}")
print(f"  Materials   : {MATERIALS_DIR}")


In [ ]:
import subprocess
import sys

GIT_URL = "https://github.com/ios-community/dzul-get-nfi.git"


def _clone_repo(repo_root, git_url):
    """Clone the repository if the target directory does not exist.

    Args:
        repo_root: The expected repository root path.
        git_url: The remote git URL to clone from.

    Raises:
        RuntimeError: If the clone command fails.
    """
    if repo_root.exists() and (repo_root / "Cargo.toml").exists():
        print(f"  Repository already present at {repo_root}")
        return

    print(f"  Cloning {git_url} ...")
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    result = subprocess.run(
        ["git", "clone", "--depth=1", git_url, str(repo_root)],
        capture_output=True,
        text=True,
        env=env,
    )
    if result.returncode != 0:
        stderr = result.stderr.strip()
        if "could not read Username" in stderr or "Authentication failed" in stderr:
            print("  Authentication required. Attempting to use stored credentials...")
            result = subprocess.run(
                ["git", "clone", "--depth=1",
                 "https://oauth2:" + os.environ.get("GITHUB_TOKEN", "") + "@github.com/ios-community/dzul-get-nfi.git"
                 if os.environ.get("GITHUB_TOKEN") else git_url,
                 str(repo_root)],
                capture_output=True, text=True, env=env,
            )
            if result.returncode != 0:
                msg = f"git clone failed even with token:\n{result.stderr}"
                raise RuntimeError(msg)
        else:
            msg = f"git clone failed:\n{stderr}"
            raise RuntimeError(msg)
    print("  Clone complete.")


if IS_CLOUD:
    _clone_repo(REPO_ROOT, GIT_URL)
    os.chdir(str(REPO_ROOT))
    sys.path.insert(0, str(REPO_ROOT / "scripts"))
    print(f"  Changed working directory to {REPO_ROOT}")
else:
    print("  Local run -- clone skipped.")
    sys.path.insert(0, str(REPO_ROOT / "scripts"))


In [ ]:
import shutil
import subprocess


def _install_python_deps(repo_root):
    """Install required Python packages via pip.

    Args:
        repo_root: The project root directory.

    Raises:
        RuntimeError: If pip install fails.
    """
    required = {"psutil", "scipy", "matplotlib", "pandas", "numpy", "jinja2"}
    installed = {pkg for pkg in required if _importable(pkg)}
    missing = required - installed

    if not missing:
        print("  All Python dependencies are already installed.")
        return

    print(f"  Installing missing packages: {', '.join(sorted(missing))}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", *sorted(missing)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        msg = f"pip install failed:\n{result.stderr}"
        raise RuntimeError(msg)


def _importable(package_name):
    """Check whether a Python package can be imported.

    Args:
        package_name: The importable name of the package.

    Returns:
        True if the import succeeds, False otherwise.
    """
    try:
        __import__(package_name)
        return True
    except ImportError:
        return False


def _install_rust():
    """Install the Rust toolchain via rustup if rustc is not on PATH.

    Raises:
        RuntimeError: If rustup-init fails.
    """
    if shutil.which("rustc") is not None:
        print(f"  Rust already installed: {_rustc_version()}")
        return

    print("  Rust not found. Installing via rustup ...")
    install_result = subprocess.run(
        ["sh", "-c", "curl --proto =https --tlsv1.2 -sSf "
         "https://sh.rustup.rs | sh -s -- -y"],
        capture_output=True,
        text=True,
    )
    if install_result.returncode != 0:
        msg = f"rustup install failed:\n{install_result.stderr}"
        raise RuntimeError(msg)

    # Add cargo to PATH for the remainder of this session
    cargo_bin = Path.home() / ".cargo" / "bin"
    if cargo_bin.exists():
        os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

    print(f"  Rust installed: {_rustc_version()}")


def _rustc_version():
    """Return the installed rustc version string.

    Returns:
        The version string (e.g. ``"1.97.1"``), or ``"unknown"``.
    """
    result = subprocess.run(
        ["rustc", "--version"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        return result.stdout.strip()
    return "unknown"


_install_python_deps(REPO_ROOT)
_install_rust()


In [ ]:
import subprocess
import sys

FORCE_RERUN = False  # Set to True to re-run even if materials exist

results_present = (
    (MATERIALS_DIR / "raw_statistics_output.txt").exists()
    and (MATERIALS_DIR / "raw_benches_output.txt").exists()
)

if results_present and not FORCE_RERUN:
    print("  Benchmark results already cached in", MATERIALS_DIR)
    print("  Set FORCE_RERUN = True above to re-run the pipeline.")
else:
    """Execute the complete benchmark pipeline via the authoritative script.

    The pipeline is run as a subprocess to avoid import conflicts with the
    notebook environment and to ensure the script's ``sys.argv`` handling
    works correctly.  Output is streamed in real time.
    """
    script_path = str(REPO_ROOT / "scripts" / "dzul_get_nfi_bench.py")
    print(f"  Running: python {script_path}")
    print("=" * 60)

    result = subprocess.run(
        [sys.executable, script_path],
        cwd=str(REPO_ROOT),
    )

    if result.returncode != 0:
        msg = (
            f"Pipeline exited with code {result.returncode}. "
            f"Check the output above for details."
        )
        raise RuntimeError(msg)

    print("=" * 60)
    print("Pipeline complete.  All results saved to", MATERIALS_DIR)
    print("=" * 60)


In [ ]:
hardware_specs = MATERIALS_DIR / "hardware_specs.md"
if hardware_specs.exists():
    print("=== Hardware Specifications ===\n")
    print(hardware_specs.read_text())
else:
    print("Hardware specs not found -- run the pipeline cell first.")


In [ ]:
stats_path = MATERIALS_DIR / "raw_statistics_output.txt"
if not stats_path.exists():
    print("Statistics output not found -- run the pipeline cell first.")
else:
    content = stats_path.read_text(encoding="utf-8")
    lines = content.split("\n")
    in_table = False
    for line in lines:
        if any(marker in line for marker in
               ["=== 2-Opt Ablation Study ===",
                "=== GROUP 1:",
                "=== GROUP 2:"]):
            in_table = True
        elif line.startswith("test ") and "ok" in line:
            in_table = False
        elif line.startswith("test result: ok"):
            in_table = False
        if in_table:
            print(line)
    print("\n--- Full output:", stats_path, "---")


In [ ]:
benches_path = MATERIALS_DIR / "raw_benches_output.txt"
if not benches_path.exists():
    print("Bench output not found -- run the pipeline cell first.")
else:
    print(benches_path.read_text(encoding="utf-8"))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

csv_path = MATERIALS_DIR / "divan_microbenchmarks.csv"

if not csv_path.exists():
    print("Microbenchmark CSV not found -- run the pipeline cell first.")
else:
    df = pd.read_csv(str(csv_path))

    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
    })

    df_alpha = df[df["Category"] == "sensitivity_threshold_alpha"].copy()
    df_c = df[df["Category"] == "sensitivity_backtrack_factor"].copy()

    if not df_alpha.empty:
        df_alpha["Param"] = df_alpha["Item"].astype(float)
        df_alpha = df_alpha.sort_values("Param")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 3.8), dpi=300)

        ax1.plot(df_alpha["Param"], df_alpha["Mean_MS"], "b-o", linewidth=1.5)
        ax1.set_xlabel(r"Geometric Threshold Multiplier ($\alpha$)")
        ax1.set_ylabel("Mean Execution Time (ms)", color="b")
        ax1.tick_params(axis="y", labelcolor="b")
        ax1.grid(True, linestyle="--", alpha=0.5)
        ax1.set_title(r"(a) Sensitivity to Threshold Multiplier $\alpha$")

        if not df_c.empty:
            df_c["Param"] = df_c["Item"].astype(int)
            df_c = df_c.sort_values("Param")
            ax2.plot(df_c["Param"], df_c["Mean_MS"], "g-o", linewidth=1.5)
            ax2.set_xlabel(r"Dynamic Backtrack Factor ($c$)")
            ax2.set_ylabel("Mean Execution Time (ms)", color="g")
            ax2.tick_params(axis="y", labelcolor="g")
            ax2.grid(True, linestyle="--", alpha=0.5)
            ax2.set_title(r"(b) Sensitivity to Backtrack Factor $c$")

        plt.tight_layout()
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(str(PLOTS_DIR / "fig_sensitivity_analysis.png"),
                    bbox_inches="tight")
        plt.savefig(str(PLOTS_DIR / "fig_sensitivity_analysis.pdf"),
                    bbox_inches="tight")
        plt.show()
        print("Figures saved to", PLOTS_DIR)
    else:
        print("Sensitivity data not found in CSV.")


In [ ]:
import shutil
import zipfile

ZIP_PATH = REPO_ROOT / "scripts" / "get_nfi_results.zip"


def _package_results(zip_path, materials_dir, plots_dir):
    """Create a zip archive of benchmark results.

    Args:
        zip_path: Destination path for the zip archive.
        materials_dir: Directory containing raw benchmark outputs.
        plots_dir: Directory containing generated figures.

    Raises:
        FileNotFoundError: If either input directory is missing.
    """
    if not materials_dir.exists():
        raise FileNotFoundError(
            f"Materials directory not found: {materials_dir}")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for src_dir, arc_prefix in [(materials_dir, "materials"),
                                     (plots_dir, "plots")]:
            if not src_dir.exists():
                continue
            for file_path in src_dir.rglob("*"):
                if file_path.is_file():
                    rel = file_path.relative_to(src_dir.parent)
                    zf.write(file_path, f"{arc_prefix}/{rel}")

    size_mb = zip_path.stat().st_size / 1_048_576
    print(f"  Created {zip_path} ({size_mb:.1f} MB)")


try:
    _package_results(ZIP_PATH, MATERIALS_DIR, PLOTS_DIR)
except FileNotFoundError as exc:
    print(f"  {exc}")
    print("  Run the pipeline cell first.")

if IS_CLOUD:
    print("\n  Download via platform file browser:")
    print(f"    from google.colab import files")
    print(f"    files.download('{ZIP_PATH}')")
